### multiple_docking_simulation.py 수행 후 docked 파일에서 affinity 파싱 ###

In [5]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd

DOCK_DIR = Path("/home/jeongin/eupatilin/data/negative_protein/GR/docking_results")

def parse_affinities_from_pdbqt(pose_path: Path):
    """
    smina/ vina 결과 pdbqt에서
    REMARK VINA RESULT 줄들을 모두 읽어서
    (pose_index, affinity) 리스트로 반환.
    """
    affinities = []
    pose_idx = 1

    with open(pose_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if "minimizedAffinity" in line:
                parts = line.strip().split()
                aff = None
                for tok in reversed(parts):
                    try:
                        aff = float(tok)
                        break
                    except ValueError:
                        continue
                if aff is not None:
                    affinities.append((pose_idx, aff))
                    pose_idx += 1
    return affinities

def main():
    rows = []

    # *_docked.pdbqt 파일들 전부 순회
    for pose_path in sorted(DOCK_DIR.glob("*_docked.pdbqt")):
        # 파일명에서 pdb_id 추출: 예) 3E7O_docked.pdbqt → 3E7O
        pdb_id = pose_path.name.split("_")[0]

        aff_list = parse_affinities_from_pdbqt(pose_path)
        if not aff_list:
            print(f"[WARN] minimizedAffinity 정보 없음: {pose_path.name}")
        for pose_idx, aff in aff_list:
            rows.append({
                "pdb_id": pdb_id,
                "pose_index": pose_idx,   # 1 = best pose
                "affinity": aff,
                "file": str(pose_path),
            })

    if not rows:
        print("추출된 affinity가 없습니다.")
        return

    df = pd.DataFrame(rows)
    out_csv = DOCK_DIR / "GR_docking_affinities.csv"
    df.to_csv(out_csv, index=False)
    print("✓ 저장 완료:", out_csv)

if __name__ == "__main__":
    main()


✓ 저장 완료: /home/jeongin/eupatilin/data/negative_protein/GR/docking_results/GR_docking_affinities.csv


In [6]:
JNKdocking = pd.read_csv("/home/jeongin/eupatilin/data/negative_protein/GR/docking_results/GR_docking_affinities.csv")
JNKdocking.to_excel("/home/jeongin/eupatilin/data/negative_protein/GR/docking_results/GR_docking_affinities.xlsx", index=False)